In [1]:
import os
import gc
import json
import pickle
import warnings
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

!pip install mlflow dagshub --quiet

import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 584.5 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 34.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## 0.1 MLflow / DagsHub Setup

Before running this notebook in Kaggle, add these Kaggle Secrets:

- `DAGSHUB_USERNAME`
- `DAGSHUB_TOKEN`

Then replace `YOUR_DAGSHUB_USERNAME` and `YOUR_REPOSITORY_NAME` below.

In [2]:
import dagshub
dagshub.init(repo_owner='ChorniBero15', repo_name='ML2', mlflow=True)

mlflow.set_experiment("RandomForest_Training")

REGISTERED_MODEL_NAME = "model_random_forest"

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=68be1909-5d6d-4f13-b2a2-8aacf3f9116c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1815985bd0cdab54cc3f99aa636d33a5433ae4afa092f00af50216e7ded2b432




Accessing as ChorniBero15

Initialized MLflow to track repo "ChorniBero15/ML2"

Repository ChorniBero15/ML2 initialized!

# 1. Cleaning

## 1.1 Data paths

In [3]:
train_transaction_path = "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv"
test_transaction_path = "/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv"
train_identity_path = "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv"
test_identity_path = "/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv"

## 1.2 Memory reduction function

The IEEE-CIS dataset is large, so numeric columns are downcasted to smaller numeric types where possible.

In [4]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    reduction = 100 * (start_mem - end_mem) / start_mem

    print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    print(f"Reduced by {reduction:.2f}%")

    return df

## 1.3 Load, clean column names, and merge data

The transaction and identity tables are merged with a left join on `TransactionID`. A left join preserves all transactions even if identity information is missing.

In [5]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):
    train_transaction = reduce_mem_usage(pd.read_csv(train_transaction_path))
    test_transaction = reduce_mem_usage(pd.read_csv(test_transaction_path))

    train_identity = reduce_mem_usage(pd.read_csv(train_identity_path))
    test_identity = reduce_mem_usage(pd.read_csv(test_identity_path))

    test_transaction.columns = test_transaction.columns.str.replace("-", "_", regex=False)
    test_identity.columns = test_identity.columns.str.replace("-", "_", regex=False)

    train = train_transaction.merge(train_identity, on="TransactionID", how="left")
    test = test_transaction.merge(test_identity, on="TransactionID", how="left")

    train = reduce_mem_usage(train)
    test = reduce_mem_usage(test)

    mlflow.log_metric("train_rows", train.shape[0])
    mlflow.log_metric("train_columns", train.shape[1])
    mlflow.log_metric("test_rows", test.shape[0])
    mlflow.log_metric("test_columns", test.shape[1])
    mlflow.log_metric("fraud_rate", train["isFraud"].mean())
    mlflow.log_metric("train_missing_percent", train.isnull().mean().mean())
    mlflow.log_metric("test_missing_percent", test.isnull().mean().mean())

    del train_transaction, test_transaction, train_identity, test_identity
    gc.collect()

print("Train:", train.shape)
print("Test:", test.shape)

Memory: 2062.07 MB -> 1203.22 MB
Reduced by 41.65%
Memory: 1771.84 MB -> 1038.31 MB
Reduced by 41.40%
Memory: 143.14 MB -> 129.94 MB
Reduced by 9.22%
Memory: 140.08 MB -> 127.09 MB
Reduced by 9.27%
Memory: 1603.31 MB -> 1603.31 MB
Reduced by 0.00%
Memory: 1386.12 MB -> 1386.12 MB
Reduced by 0.00%
🏃 View run RandomForest_Cleaning at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/02c9583e44104d5e804b6faf471bd2ec
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2
Train: (590540, 434)
Test: (506691, 433)


# 2. Feature Engineering

## 2.1 Pipeline transformers

These custom transformers make the final model a complete sklearn `Pipeline`. The pipeline can receive the raw merged test dataframe and apply all preprocessing internally.

In [6]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols if cols is not None else []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        existing_cols = [col for col in self.cols if col in X.columns]
        return X.drop(columns=existing_cols)


class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.created_cols = [
            "TransactionAmt_log",
            "TransactionAmt_decimal",
            "Transaction_day",
            "Transaction_hour",
            "Transaction_week",
            "missing_count",
            "has_identity",
            "email_domain_match",
            "card_missing_count"
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[col for col in self.created_cols if col in X.columns], errors="ignore")

        features = pd.DataFrame(index=X.index)

        if "TransactionAmt" in X.columns:
            features["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            features["TransactionAmt_decimal"] = ((X["TransactionAmt"] % 1) * 1000).round()

        if "TransactionDT" in X.columns:
            features["Transaction_day"] = X["TransactionDT"] // 86400
            features["Transaction_hour"] = (X["TransactionDT"] // 3600) % 24
            features["Transaction_week"] = features["Transaction_day"] // 7

        features["missing_count"] = X.isnull().sum(axis=1)

        if "id_01" in X.columns:
            features["has_identity"] = X["id_01"].notnull().astype("int8")
        else:
            features["has_identity"] = 0

        if "P_emaildomain" in X.columns and "R_emaildomain" in X.columns:
            features["email_domain_match"] = (X["P_emaildomain"] == X["R_emaildomain"]).astype("int8")
        else:
            features["email_domain_match"] = 0

        card_cols = [col for col in ["card1", "card2", "card3", "card4", "card5", "card6"] if col in X.columns]

        if len(card_cols) > 0:
            features["card_missing_count"] = X[card_cols].isnull().sum(axis=1)
        else:
            features["card_missing_count"] = 0

        return pd.concat([X, features], axis=1)


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.freq_maps = {}

    def _normalize_series(self, s):
        return s.astype("object").where(s.notnull(), "__MISSING__")

    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object", "category"]).columns.tolist()
        else:
            self.cols_ = [col for col in self.cols if col in X.columns]

        self.freq_maps = {}

        for col in self.cols_:
            normalized = self._normalize_series(X[col])
            self.freq_maps[col] = normalized.value_counts(dropna=False).to_dict()

        return self

    def transform(self, X):
        X = X.copy()
        new_features = pd.DataFrame(index=X.index)

        for col in self.cols_:
            if col in X.columns:
                normalized = self._normalize_series(X[col])
                new_features[f"{col}_freq"] = normalized.map(self.freq_maps[col]).fillna(0).astype("float32")

        return pd.concat([X, new_features], axis=1)


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.category_maps = {}

    def _normalize_series(self, s):
        return s.astype("object").where(s.notnull(), "__MISSING__")

    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object", "category"]).columns.tolist()
        else:
            self.cols_ = [col for col in self.cols if col in X.columns]

        self.category_maps = {}

        for col in self.cols_:
            normalized = self._normalize_series(X[col])
            unique_values = pd.Series(normalized.unique())
            self.category_maps[col] = {value: idx for idx, value in enumerate(unique_values)}

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.cols_:
            if col in X.columns:
                normalized = self._normalize_series(X[col])
                X[col] = normalized.map(self.category_maps[col]).fillna(-1).astype("int32")

        return X


class ReplaceInfValues(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.replace([np.inf, -np.inf], np.nan)

# 3. Feature Selection

## 3.1 Feature selection and imputation transformers

Random Forest cannot handle NaN values directly, so median imputation is included inside the pipeline.

In [7]:
class SimpleFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, max_missing_ratio=0.95, min_unique_values=2):
        self.max_missing_ratio = max_missing_ratio
        self.min_unique_values = min_unique_values

    def fit(self, X, y=None):
        X = X.copy()

        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[numeric_cols]

        missing_ratio = X_numeric.isnull().mean()
        unique_counts = X_numeric.nunique(dropna=False)

        self.selected_features_ = [
            col for col in numeric_cols
            if missing_ratio[col] <= self.max_missing_ratio
            and unique_counts[col] >= self.min_unique_values
        ]

        self.dropped_features_ = [col for col in numeric_cols if col not in self.selected_features_]

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.selected_features_:
            if col not in X.columns:
                X[col] = np.nan

        return X[self.selected_features_]


class TopCorrelationFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=180, sample_size=120000, random_state=42):
        self.max_features = max_features
        self.sample_size = sample_size
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.copy()

        if y is None:
            self.selected_features_ = X.select_dtypes(include=[np.number]).columns.tolist()
            return self

        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[numeric_cols]
        y_series = pd.Series(y, index=X.index)

        if len(X_numeric) > self.sample_size:
            sample_index = X_numeric.sample(
                n=self.sample_size,
                random_state=self.random_state
            ).index

            X_numeric = X_numeric.loc[sample_index]
            y_series = y_series.loc[sample_index]

        medians = X_numeric.median(numeric_only=True)
        X_filled = X_numeric.fillna(medians)

        correlations = X_filled.corrwith(y_series).abs().fillna(0)
        correlations = correlations.sort_values(ascending=False)

        self.feature_scores_ = correlations.to_dict()
        self.selected_features_ = correlations.head(self.max_features).index.tolist()

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.selected_features_:
            if col not in X.columns:
                X[col] = np.nan

        return X[self.selected_features_]


class MedianImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()
        self.columns_ = X.columns.tolist()
        self.medians_ = X.median(numeric_only=True)
        self.medians_ = self.medians_.fillna(0)
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.columns_:
            if col not in X.columns:
                X[col] = np.nan

        X = X[self.columns_]
        X = X.fillna(self.medians_)

        return X

## 3.2 Prepare train / validation data

A time-based split is used because `TransactionDT` is a time-like variable and the test set comes after the training period.

In [8]:
target = "isFraud"

X = train.drop(columns=[target])
y = train[target].astype("int8")

X_test = test.copy()

split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

X_val = X.iloc[split_index:].copy()
y_val = y.iloc[split_index:].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())

X_train: (472432, 433)
X_val: (118108, 433)
Train fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145


## 3.3 Frequency encoding columns

In [9]:
freq_encode_cols = [
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo",
    "ProductCD",
    "id_30", "id_31", "id_33"
]

## 3.4 Log Feature Engineering run

In [10]:
with mlflow.start_run(run_name="RandomForest_Feature_Engineering"):
    feature_engineering_pipeline = Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues())
    ])

    feature_engineering_pipeline.fit(X_train, y_train)
    X_sample_fe = feature_engineering_pipeline.transform(X_train.iloc[:1000])

    mlflow.log_param("engineered_features", "amount,time,missingness,identity,email_match,frequency_encoding")
    mlflow.log_metric("features_after_engineering", X_sample_fe.shape[1])

    print("Features after engineering:", X_sample_fe.shape[1])

Features after engineering: 457
🏃 View run RandomForest_Feature_Engineering at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/ff07496a1f024c1da38e54ace399bd89
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2


## 3.5 Log Feature Selection run

In [11]:
with mlflow.start_run(run_name="RandomForest_Feature_Selection"):
    feature_selection_pipeline = Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues()),
        ("simple_feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
        ("top_correlation_selection", TopCorrelationFeatureSelector(max_features=180, sample_size=120000)),
        ("median_imputer", MedianImputer())
    ])

    feature_selection_pipeline.fit(X_train, y_train)

    simple_selector = feature_selection_pipeline.named_steps["simple_feature_selection"]
    top_selector = feature_selection_pipeline.named_steps["top_correlation_selection"]

    selected_features = top_selector.selected_features_

    with open("random_forest_selected_features.json", "w") as f:
        json.dump(selected_features, f, indent=2)

    with open("random_forest_feature_scores.json", "w") as f:
        json.dump(top_selector.feature_scores_, f, indent=2)

    mlflow.log_param("feature_selection_method", "missing_constant_filter_plus_top_correlation")
    mlflow.log_param("max_missing_ratio", 0.95)
    mlflow.log_param("top_correlation_max_features", 180)
    mlflow.log_metric("features_after_simple_selection", len(simple_selector.selected_features_))
    mlflow.log_metric("features_after_top_correlation_selection", len(selected_features))
    mlflow.log_artifact("random_forest_selected_features.json")
    mlflow.log_artifact("random_forest_feature_scores.json")

    print("Features after simple selection:", len(simple_selector.selected_features_))
    print("Features after top correlation selection:", len(selected_features))

Features after simple selection: 450
Features after top correlation selection: 180
🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/104548cf18ea4ba0831f92a6ade541be
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2


# 4. Training

## 4.1 Pipeline builder

In [12]:
def build_random_forest_pipeline(rf_params):
    return Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues()),
        ("simple_feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
        ("top_correlation_selection", TopCorrelationFeatureSelector(max_features=180, sample_size=120000)),
        ("median_imputer", MedianImputer()),
        ("model", RandomForestClassifier(**rf_params))
    ])

## 4.2 Baseline Random Forest

In [13]:
baseline_rf_params = {
    "n_estimators": 150,
    "max_depth": 16,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "random_state": 42,
    "n_jobs": -1
}

with mlflow.start_run(run_name="RandomForest_Baseline"):
    baseline_pipeline = build_random_forest_pipeline(baseline_rf_params)
    baseline_pipeline.fit(X_train, y_train)

    train_pred_proba = baseline_pipeline.predict_proba(X_train)[:, 1]
    val_pred_proba = baseline_pipeline.predict_proba(X_val)[:, 1]

    baseline_train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    baseline_val_roc_auc = roc_auc_score(y_val, val_pred_proba)
    baseline_train_pr_auc = average_precision_score(y_train, train_pred_proba)
    baseline_val_pr_auc = average_precision_score(y_val, val_pred_proba)
    baseline_overfit_gap = baseline_train_roc_auc - baseline_val_roc_auc

    mlflow.log_params(baseline_rf_params)
    mlflow.log_param("model_architecture", "RandomForest")
    mlflow.log_param("validation_strategy", "time_based_80_20_split")
    mlflow.log_metric("train_roc_auc", baseline_train_roc_auc)
    mlflow.log_metric("validation_roc_auc", baseline_val_roc_auc)
    mlflow.log_metric("train_pr_auc", baseline_train_pr_auc)
    mlflow.log_metric("validation_pr_auc", baseline_val_pr_auc)
    mlflow.log_metric("overfit_gap", baseline_overfit_gap)

    print("Baseline train ROC-AUC:", baseline_train_roc_auc)
    print("Baseline validation ROC-AUC:", baseline_val_roc_auc)
    print("Baseline overfit gap:", baseline_overfit_gap)

Baseline train ROC-AUC: 0.9118430374677348
Baseline validation ROC-AUC: 0.8497296959928022
Baseline overfit gap: 0.062113341474932526
🏃 View run RandomForest_Baseline at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/2e2de19919094920b9438983fbdba109
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2


## 4.3 Grid Search Random Forest

This grid is intentionally small. Random Forest is slower and heavier than XGBoost on this dataset.

In [14]:
rf_base_params = {
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "random_state": 42,
    "n_jobs": -1
}

rf_param_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [12, 20],
    "model__min_samples_split": [20],
    "model__min_samples_leaf": [5, 15],
    "model__max_features": ["sqrt"]
}

candidate_count = 1
for values in rf_param_grid.values():
    candidate_count *= len(values)

print("Grid candidate count:", candidate_count)

grid_search_pipeline = build_random_forest_pipeline(rf_base_params)

time_cv = TimeSeriesSplit(n_splits=2)

grid_search = GridSearchCV(
    estimator=grid_search_pipeline,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=time_cv,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True
)

with mlflow.start_run(run_name="RandomForest_Grid_Search"):
    grid_search.fit(X_train, y_train)

    grid_results_df = pd.DataFrame(grid_search.cv_results_)
    grid_results_df.to_csv("random_forest_grid_search_results.csv", index=False)

    train_pred_proba = grid_search.best_estimator_.predict_proba(X_train)[:, 1]
    val_pred_proba = grid_search.best_estimator_.predict_proba(X_val)[:, 1]

    grid_train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    grid_val_roc_auc = roc_auc_score(y_val, val_pred_proba)
    grid_train_pr_auc = average_precision_score(y_train, train_pred_proba)
    grid_val_pr_auc = average_precision_score(y_val, val_pred_proba)
    grid_overfit_gap = grid_train_roc_auc - grid_val_roc_auc

    with open("random_forest_param_grid.json", "w") as f:
        json.dump(rf_param_grid, f, indent=2)

    mlflow.log_param("search_method", "GridSearchCV")
    mlflow.log_param("cv_strategy", "TimeSeriesSplit")
    mlflow.log_param("cv_splits", time_cv.n_splits)
    mlflow.log_param("scoring", "roc_auc")
    mlflow.log_param("candidate_count", candidate_count)

    for param_name, param_value in grid_search.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)

    mlflow.log_metric("best_cv_roc_auc", grid_search.best_score_)
    mlflow.log_metric("grid_train_roc_auc", grid_train_roc_auc)
    mlflow.log_metric("grid_validation_roc_auc", grid_val_roc_auc)
    mlflow.log_metric("grid_train_pr_auc", grid_train_pr_auc)
    mlflow.log_metric("grid_validation_pr_auc", grid_val_pr_auc)
    mlflow.log_metric("grid_overfit_gap", grid_overfit_gap)

    mlflow.log_artifact("random_forest_param_grid.json")
    mlflow.log_artifact("random_forest_grid_search_results.csv")

best_params_from_grid = rf_base_params.copy()

for param_name, param_value in grid_search.best_params_.items():
    clean_name = param_name.replace("model__", "")
    best_params_from_grid[clean_name] = param_value

rf_pipeline = grid_search.best_estimator_
best_params_for_final = best_params_from_grid.copy()

grid_metrics = {
    "run_name": "RandomForest_Grid_Search",
    "best_cv_roc_auc": grid_search.best_score_,
    "train_roc_auc": grid_train_roc_auc,
    "validation_roc_auc": grid_val_roc_auc,
    "train_pr_auc": grid_train_pr_auc,
    "validation_pr_auc": grid_val_pr_auc,
    "overfit_gap": grid_overfit_gap
}

print("Best grid search params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)
print("Grid validation ROC-AUC:", grid_val_roc_auc)
print("Grid validation PR-AUC:", grid_val_pr_auc)
print("Grid overfit gap:", grid_overfit_gap)

Grid candidate count: 8
Fitting 2 folds for each of 8 candidates, totalling 16 fits
[CV] END model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=20, model__n_estimators=150; total time=  26.8s
[CV] END model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=20, model__n_estimators=150; total time=  48.5s
[CV] END model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=20, model__n_estimators=250; total time=  36.7s
[CV] END model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=20, model__n_estimators=250; total time= 1.2min
[CV] END model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=15, model__min_samples_split=20, model__n_estimators=150; total time=  26.4s
[CV] END model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=15, model__min_samples_split=20, model__n_estimators=15

## 4.4 Compare baseline and grid search

In [15]:
results_df = pd.DataFrame([
    {
        "run_name": "RandomForest_Baseline",
        "validation_roc_auc": baseline_val_roc_auc,
        "validation_pr_auc": baseline_val_pr_auc,
        "train_roc_auc": baseline_train_roc_auc,
        "train_pr_auc": baseline_train_pr_auc,
        "overfit_gap": baseline_overfit_gap
    },
    {
        "run_name": "RandomForest_Grid_Search",
        "validation_roc_auc": grid_val_roc_auc,
        "validation_pr_auc": grid_val_pr_auc,
        "train_roc_auc": grid_train_roc_auc,
        "train_pr_auc": grid_train_pr_auc,
        "overfit_gap": grid_overfit_gap
    }
])

results_df = results_df.sort_values("validation_roc_auc", ascending=False)
results_df.to_csv("random_forest_experiment_results.csv", index=False)

results_df

,run_name,validation_roc_auc,validation_pr_auc,train_roc_auc,train_pr_auc,overfit_gap
1,RandomForest_Grid_Search,0.854057,0.413003,0.925213,0.625596,0.071155
0,RandomForest_Baseline,0.849730,0.407283,0.911843,0.607607,0.062113


In [16]:
with mlflow.start_run(run_name="RandomForest_Model_Comparison"):
    mlflow.log_artifact("random_forest_experiment_results.csv")

    mlflow.log_param("best_run_name", results_df.iloc[0]["run_name"])
    mlflow.log_metric("best_validation_roc_auc", results_df.iloc[0]["validation_roc_auc"])
    mlflow.log_metric("best_validation_pr_auc", results_df.iloc[0]["validation_pr_auc"])
    mlflow.log_metric("best_overfit_gap", results_df.iloc[0]["overfit_gap"])

🏃 View run RandomForest_Model_Comparison at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/cba8896ca2f649d09de6670320bcdd73
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2


## 4.5 Select best parameters

In [17]:
best_run_name = results_df.iloc[0]["run_name"]

if best_run_name == "RandomForest_Baseline":
    best_params_for_final = baseline_rf_params.copy()
else:
    best_params_for_final = best_params_from_grid.copy()

print("Best run:", best_run_name)
print(best_params_for_final)

Best run: RandomForest_Grid_Search
{'bootstrap': True, 'class_weight': 'balanced_subsample', 'random_state': 42, 'n_jobs': -1, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 15, 'min_samples_split': 20, 'n_estimators': 250}


## 4.6 Train final pipeline on full training data and register model

In [18]:
final_pipeline = build_random_forest_pipeline(best_params_for_final)

with mlflow.start_run(run_name="RandomForest_Final_Pipeline"):
    final_pipeline.fit(X, y)

    full_train_pred = final_pipeline.predict_proba(X)[:, 1]

    full_train_roc_auc = roc_auc_score(y, full_train_pred)
    full_train_pr_auc = average_precision_score(y, full_train_pred)

    mlflow.log_params(best_params_for_final)
    mlflow.log_param("selected_best_run", best_run_name)
    mlflow.log_param("model_saved_as", "sklearn_pipeline")
    mlflow.log_metric("full_train_roc_auc", full_train_roc_auc)
    mlflow.log_metric("full_train_pr_auc", full_train_pr_auc)

    with open("model.pkl", "wb") as f:
        pickle.dump(final_pipeline, f)

    mlflow.log_artifact("model.pkl")
    mlflow.log_artifact("random_forest_experiment_results.csv")

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME
    )

    print("Final Random Forest pipeline saved and registered.")
    print("Full train ROC-AUC:", full_train_roc_auc)
    print("Full train PR-AUC:", full_train_pr_auc)

2026/05/06 15:53:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:53:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'model_random_forest' already exists. Creating a new version of this model...
2026/05/06 15:53:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_random_forest, version 2
Created version '2' of model 'model_random_forest'.


Final Random Forest pipeline saved and registered.
Full train ROC-AUC: 0.9353917247544478
Full train PR-AUC: 0.6325307603186948
🏃 View run RandomForest_Final_Pipeline at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/abeb7dc00ddc4f3b96857e59b604738c
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2


# 5. Submission

## 5.1 Create submission from final pipeline

In [19]:
test_ids = X_test["TransactionID"].copy()

y_proba = final_pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": y_proba
})

submission.to_csv("submission_random_forest.csv", index=False)

submission.head()

,TransactionID,isFraud
0,3663549,0.114377
1,3663550,0.269262
2,3663551,0.263239
3,3663552,0.074071
4,3663553,0.054082


In [20]:
with mlflow.start_run(run_name="RandomForest_Submission"):
    mlflow.log_artifact("submission_random_forest.csv")
    mlflow.log_metric("submission_rows", submission.shape[0])

🏃 View run RandomForest_Submission at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2/runs/4bb637d8f3ea467b97abff6961f712d2
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/2


# 6. Notes for README

Random Forest is expected to be slower and may perform worse than XGBoost on this dataset. That is acceptable for this assignment because the goal is not only leaderboard score. The goal is to test different model architectures, log experiments, show underfit/overfit behavior, and justify the final model selection.

Key points to mention in README:

- Random Forest required median imputation because sklearn RandomForest cannot handle NaN directly.
- Feature selection was more aggressive than XGBoost to reduce training time.
- TimeSeriesSplit was used because `TransactionDT` is time-like.
- Grid search was intentionally small due to memory and runtime constraints.
- Final model was saved as a full sklearn Pipeline and registered in MLflow Model Registry.